# Construction de la base de données — Registres de la Chancellerie royale (AN JJ)

 Ce script transforme trois fichiers CSV sources en quatre tables normalisées importables dans une base de données relationnelle :

 | Table | Description |
 |---|---|
 | `images.csv` | Une ligne par image numérisée |
 | `zones.csv` | Une ligne par zone YOLO détectée |
 | `actes.csv` | Une ligne par acte avec concordances aplaties |
 | `actes_images_zones.csv` | Liaisons acte ↔ image ↔ zone, avec rôles et statuts |

 **Structure valide d'un acte :**
 - `[AC]` — acte complet, zone unique
 - `[AI] + [AF]` — acte initial + final
 - `[AI] + [AM]⁺ + [AF]` — avec une ou plusieurs zones médianes (pages entières)



# Paramètres


In [ ]:
import os
import re

corpus = 'JJ096-JJ099'





# Chemins des fichiers sources
INPUT_ZONES  = f"../List-of-zones/Himanis_Seg_Actes_1200pxmin_{corpus}_labelstudio.csv"       # fichier 1 : zones YOLO
INPUT_ACTES  = "../List-of-acts/Acts-JJ96-JJ211.csv"                # fichier 2 : liste des actes
INPUT_IMAGES = f"../List-of-images/{corpus}_image_data.csv"    # fichier 3 : images téléchargées

# Dossier de sortie
OUT_DIR = f"{corpus}/output"
os.makedirs(OUT_DIR, exist_ok=True)

# Labels de zones attendus par le modèle d'acte
LABELS_ATTENDUS = {'AC', 'AI', 'AM', 'AF', 'NIA', 'Table'}



def parse_corpus(corpus):
    match = re.match(r'([A-Z]+)(\d+)-([A-Z]+)(\d+)', corpus)
    if not match:
        raise ValueError(f"Format inattendu : {corpus}")

    prefix = match.group(1)
    start = int(match.group(2))
    end = int(match.group(4))
    width = len(match.group(2))  # conserve le nombre de chiffres initial

    return {f"{prefix}{i:0{width}d}" for i in range(start, end + 1)}


REGISTRES_CIBLES = sorted(parse_corpus(corpus))
print(REGISTRES_CIBLES)

# Séparateur CSV des fichiers sources ('\t' = tabulation)
SEP = '\t'


{'JJ098', 'JJ096', 'JJ099', 'JJ097'}


# Imports et fonctions utilitaires

In [49]:
!python --version


Python 3.12.7


In [50]:
!pip install openpyxl
!pip install pandas

In [51]:
import os
import re
import ast
import json
import warnings
import pandas as pd
import numpy as np
from collections import defaultdict


pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 50)

# %%
def normalize_path(p):
    """Normalise les séparateurs Windows/Unix et renvoie le basename."""
    return os.path.basename(str(p).replace("\\", "/"))


def extract_register(folder_path):
    """
    Extrait le numéro de registre normalisé depuis un chemin.
    'Paris_Archives_Nationales_JJ096' → 'JJ96' (sans zéro initial).
    Cible le DERNIER composant du chemin contenant JJ + chiffres.
    """
    parts = str(folder_path).replace("\\", "/").split("/")
    for part in reversed(parts):
        m = re.search(r'JJ0*(\d+)$', part)
        if m:
            return f"JJ{m.group(1)}"
    return None


def normalize_folio(raw):
    """
    Normalise un label de folio pour jointure.
      '6'        → '6r'
      '12v'      → '12v'
      '1r'       → '1r'
      '103bis'   → '103bisr'
      '103bis v' → '103bisv'
      '103 bis recto' → '103bisr'
      '103 bis verso' → '103bisv'
      'plat supérieur' → 'plat supérieur'
    """
    s = str(raw).strip().lower()

    # Dans normalize_folio, avant toute autre règle :
    if s in ('vacat', 'vacatr', 'vacatv'):
        return None   # sera logué comme "sans folio" → ignoré proprement
    
    if not s or s == 'nan':
        return None

    # Normaliser les variantes textuelles de recto/verso
    s = s.replace('recto', 'r').replace('verso', 'v')

    # Supprimer les espaces autour de 'bis' et avant r/v
    # '103 bis r' → '103bisr', '103 bis v' → '103bisv', '103 bis' → '103bis'
    s = re.sub(r'\s+bis\s*', 'bis', s)  # espaces autour de bis
    s = re.sub(r'\s+([rv])$', r'\1', s) # espace avant r/v final

    # Cas standard : chiffres + optionnel 'bis' + r ou v
    if re.match(r'^\d+(?:bis)?[rv]$', s):
        return s

    # Chiffres + optionnel 'bis' sans suffixe → recto par défaut
    if re.match(r'^\d+(?:bis)?$', s):
        return s + 'r'

    return s


def parse_abs_coords(coord_str):
    """Parse '1,23,3866,6279' → [x, y, w, h] ou None si invalide."""
    try:
        parts = [int(v) for v in str(coord_str).split(',')]
        if len(parts) == 4:
            return parts
    except Exception:
        pass
    return None

def parse_image_stem(image_path):
    stem = normalize_path(image_path).rsplit('.', 1)[0]  # retire .jpg
    parts = stem.split('_')
    volume = parts[-2]           # 'JJ096'
    folio_sort_key = int(parts[-1])  # 100
    return volume, folio_sort_key


def safe_to_int(series):
    return (
        series
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)              # ou dropna selon ton besoin
        .round()
        .astype(int)
    )

# Chargement des fichiers CSV sources

## Table `images.csv`

In [53]:
skipped = {}

with warnings.catch_warnings(record=True) as w_images:
    warnings.simplefilter("always")
    if corpus == 'JJ096-JJ099':
        encoding = 'utf-7'
    else:
        encoding = 'utf-8'
    df_images_raw = pd.read_csv(INPUT_IMAGES, sep=',', dtype=str,
                             low_memory=False, on_bad_lines='warn', encoding=encoding)
    skipped['images'] = len(w_images)

print(f"Images chargées : {len(df_images_raw):>6} lignes  ({skipped['images']} ligne(s) ignorée(s))")

df_images = df_images_raw[[
    'manifestURL', 'canvasId', 'urlImage', 'imageLabel',
    'imageFileName', 'imageWidthAsDownloaded', 'imageHeightAsDownloaded',
    'urlResizedImage', 'ResizedImageWidthAsDownloaded',
    'ResizedImageHeightAsDownloaded', 'folderPath'
]].copy()


df_images.head(5)

Images chargées :   1554 lignes  (0 ligne(s) ignorée(s))


,manifestURL,canvasId,urlImage,imageLabel,imageFileName,imageWidthAsDownloaded,imageHeightAsDownloaded,urlResizedImage,ResizedImageWidthAsDownloaded,ResizedImageHeightAsDownloaded,folderPath
0,https://api.irht.cnrs.fr/ark:/63955/fvdy3kcmb2es/manifest.json,https://arca.irht.cnrs.fr/iiif/125462/canvas/canvas-4198199,https://iiif.irht.cnrs.fr/iiif/ark:/63955/vd0qz1ihxyni/full/full/0/default.jpg,plat sup��rieur,images_registres_AN_JJ035_JJ211/images\Paris_Archives_Nationales_JJ096\Paris_Archives_Nationales_JJ096_1.jpg,4311,6605,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vd0qz1ihxyni/full/1200,/0/default.jpg",1200,1839,images_registres_AN_JJ035_JJ211/images\Paris_Archives_Nationales_JJ096
1,https://api.irht.cnrs.fr/ark:/63955/fvdy3kcmb2es/manifest.json,https://arca.irht.cnrs.fr/iiif/125462/canvas/canvas-4198200,https://iiif.irht.cnrs.fr/iiif/ark:/63955/v6e576jlpbid/full/full/0/default.jpg,contre-plat sup��rieur,images_registres_AN_JJ035_JJ211/images\Paris_Archives_Nationales_JJ096\Paris_Archives_Nationales_JJ096_2.jpg,2048,2048,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/v6e576jlpbid/full/1200,/0/default.jpg",1200,1200,images_registres_AN_JJ035_JJ211/images\Paris_Archives_Nationales_JJ096
2,https://api.irht.cnrs.fr/ark:/63955/fvdy3kcmb2es/manifest.json,https://arca.irht.cnrs.fr/iiif/125462/canvas/canvas-4198201,https://iiif.irht.cnrs.fr/iiif/ark:/63955/vxyc3rg33kh4/full/full/0/default.jpg,1r,images_registres_AN_JJ035_JJ211/images\Paris_Archives_Nationales_JJ096\Paris_Archives_Nationales_JJ096_3.jpg,3984,6437,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vxyc3rg33kh4/full/1200,/0/default.jpg",1200,1939,images_registres_AN_JJ035_JJ211/images\Paris_Archives_Nationales_JJ096
3,https://api.irht.cnrs.fr/ark:/63955/fvdy3kcmb2es/manifest.json,https://arca.irht.cnrs.fr/iiif/125462/canvas/canvas-4198202,https://iiif.irht.cnrs.fr/iiif/ark:/63955/vbg5uy3smlz0/full/full/0/default.jpg,1v,images_registres_AN_JJ035_JJ211/images\Paris_Archives_Nationales_JJ096\Paris_Archives_Nationales_JJ096_4.jpg,3768,6373,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vbg5uy3smlz0/full/1200,/0/default.jpg",1200,2030,images_registres_AN_JJ035_JJ211/images\Paris_Archives_Nationales_JJ096
4,https://api.irht.cnrs.fr/ark:/63955/fvdy3kcmb2es/manifest.json,https://arca.irht.cnrs.fr/iiif/125462/canvas/canvas-4198203,https://iiif.irht.cnrs.fr/iiif/ark:/63955/vr66brwmaaj8/full/full/0/default.jpg,2r,images_registres_AN_JJ035_JJ211/images\Paris_Archives_Nationales_JJ096\Paris_Archives_Nationales_JJ096_5.jpg,3928,6453,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vr66brwmaaj8/full/1200,/0/default.jpg",1200,1971,images_registres_AN_JJ035_JJ211/images\Paris_Archives_Nationales_JJ096


In [54]:


df_images[['volume', 'folio_sort_key']] = df_images['imageFileName'].apply(
    lambda p: pd.Series(parse_image_stem(p))
)

df_images['image_id']       = range(1, len(df_images) + 1)
df_images['image_filename'] = df_images['imageFileName'].apply(normalize_path)
df_images[['volume', 'folio_sort_key']] = df_images['image_filename'].apply(
    lambda p: pd.Series(parse_image_stem(p))
    )
df_images['register']       = df_images['folderPath'].apply(extract_register)
df_images['folio_norm']     = df_images['imageLabel'].apply(normalize_folio)

df_images.rename(columns={
    'manifestURL':                    'manifest_url',
    'canvasId':                       'canvas_id',
    'urlImage':                       'url_full',
    'imageLabel':                     'folio_label',
    'imageWidthAsDownloaded':         'width_px',
    'imageHeightAsDownloaded':        'height_px',
    'urlResizedImage':                'url_resized',
    'ResizedImageWidthAsDownloaded':  'resized_width_px',
    'ResizedImageHeightAsDownloaded': 'resized_height_px',
}, inplace=True)

df_images.drop(columns=['imageFileName', 'folderPath'], inplace=True)

IMAGES_COLS = [
    'image_id', 'register', 'image_filename', 'volume', 'folio_sort_key', 
    'folio_label', 'folio_norm',
    'width_px', 'height_px', 'url_full', 'url_resized',
    'resized_width_px', 'resized_height_px', 'manifest_url', 'canvas_id',
]
df_images = df_images[IMAGES_COLS]

df_images.to_csv(os.path.join(OUT_DIR, "images.csv"), index=False, sep=SEP)
print(f"images.csv → {len(df_images)} lignes, {len(df_images.columns)} colonnes")
df_images.head(3)

# %%
# Index (register, folio_norm) → [image_id, ...]  (ordre séquentiel préservé)
img_by_folio = defaultdict(list)
for _, row in df_images.iterrows():
    if pd.notna(row['volume']) and pd.notna(row['folio_sort_key']):
        img_by_folio[(row['volume'], int(row['folio_sort_key']))].append(row['image_id'])

        
# image_id → rang dans le registre (pour navigation séquentielle vers images suivantes)
# On construit un index par registre : register → [image_id ordonné]
images_by_register = defaultdict(list)
for _, row in df_images.iterrows():
    if row['register']:
        images_by_register[row['register']].append(row['image_id'])

print(f"Index img_by_folio    : {len(img_by_folio)} clés (register, folio_norm)")
print(f"Registres distincts   : {list(images_by_register.keys())}")


df_images.head(3)

images.csv → 1554 lignes, 15 colonnes
Index img_by_folio    : 1554 clés (register, folio_norm)
Registres distincts   : ['JJ96', 'JJ97', 'JJ98', 'JJ99']


,image_id,register,image_filename,volume,folio_sort_key,folio_label,folio_norm,width_px,height_px,url_full,url_resized,resized_width_px,resized_height_px,manifest_url,canvas_id
0,1,JJ96,Paris_Archives_Nationales_JJ096_1.jpg,JJ096,1,plat sup��rieur,plat sup��rieur,4311,6605,https://iiif.irht.cnrs.fr/iiif/ark:/63955/vd0qz1ihxyni/full/full/0/default.jpg,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vd0qz1ihxyni/full/1200,/0/default.jpg",1200,1839,https://api.irht.cnrs.fr/ark:/63955/fvdy3kcmb2es/manifest.json,https://arca.irht.cnrs.fr/iiif/125462/canvas/canvas-4198199
1,2,JJ96,Paris_Archives_Nationales_JJ096_2.jpg,JJ096,2,contre-plat sup��rieur,contre-plat sup��rieur,2048,2048,https://iiif.irht.cnrs.fr/iiif/ark:/63955/v6e576jlpbid/full/full/0/default.jpg,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/v6e576jlpbid/full/1200,/0/default.jpg",1200,1200,https://api.irht.cnrs.fr/ark:/63955/fvdy3kcmb2es/manifest.json,https://arca.irht.cnrs.fr/iiif/125462/canvas/canvas-4198200
2,3,JJ96,Paris_Archives_Nationales_JJ096_3.jpg,JJ096,3,1r,1r,3984,6437,https://iiif.irht.cnrs.fr/iiif/ark:/63955/vxyc3rg33kh4/full/full/0/default.jpg,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vxyc3rg33kh4/full/1200,/0/default.jpg",1200,1939,https://api.irht.cnrs.fr/ark:/63955/fvdy3kcmb2es/manifest.json,https://arca.irht.cnrs.fr/iiif/125462/canvas/canvas-4198201


## Table `actes.csv`

In [65]:
 

skipped = {}

with warnings.catch_warnings(record=True) as w_actes:
    warnings.simplefilter("always")
    df_actes_raw = pd.read_csv(INPUT_ACTES, sep=";")
    skipped['actes'] = len(w_actes)



print(f"Actes  chargés  : {len(df_actes_raw):>6} lignes  ({skipped['actes']} ligne(s) ignorée(s))")


df_actes = df_actes_raw.copy()

df_actes.rename(columns={
    'ID-temporaire':        'acte_id',
    'Register':             'volume',
    'Act_number':           'act_number',
    'Nvelle numérotation':  'new_numbering',
    'Folio Number ou page': 'folio_raw',
    'vérif':                'verified',
    'Note':                 'note',
}, inplace=True)



df_actes = df_actes[df_actes['volume'].isin(REGISTRES_CIBLES)].copy()
print(f"Actes après filtre {corpus} : {len(df_actes)} lignes")
print(df_actes['volume'].value_counts().sort_index())

df_actes['folio_norm'] = df_actes['folio_raw'].apply(normalize_folio)


df_actes['folio_norm'] = df_actes['folio_raw'].apply(normalize_folio)


# Table de correspondance folio_norm → folio_sort_key par registre
# On garde une seule valeur de folio_sort_key par (register, folio_norm) via first()
folio_to_sortkey = (df_images[['volume', 'folio_norm', 'folio_sort_key']]
                    .drop_duplicates(subset=['volume', 'folio_norm'])
                    .reset_index(drop=True))

# Merge à la place du apply (plus rapide, pas de problème de multi-index)
df_actes = df_actes.merge(
    folio_to_sortkey,
    on=['volume', 'folio_norm'],
    how='left'
)

# Signaler les folios non matchés
unmatched = df_actes[df_actes['folio_sort_key'].isna()]
if not unmatched.empty:
    print(f"⚠️  {len(unmatched)} acte(s) sans folio_sort_key :")
    print(unmatched[['acte_id', 'volume', 'folio_norm']].to_string(index=False))
else:
    print("✅ Tous les folios ont été matchés.")
    


# Aplatir les concordances : chaque inventaire source donne 3 colonnes
concordance_cols = [c for c in df_actes.columns if '.xml_' in c]
inventory_sources = defaultdict(list)
for col in concordance_cols:
    src = col.split('.xml_')[0] + '.xml'
    inventory_sources[src].append(col)

for src, cols in inventory_sources.items():
    short = re.sub(r'^Paris_AN_JJ_inventaire_', '', src.replace('.xml', ''))
    short = short.replace('Guerin_tome1-tome12', 'Guerin')
    for suffix, key in [('_Act_number', 'act'), ('_Head', 'head'), ('_Locus', 'locus')]:
        col = next((c for c in cols if c.endswith(suffix)), None)
        if col:
            df_actes[f'conc_{short}_{key}'] = df_actes[col].fillna('')

BASE_COLS = ['acte_id', 'volume', 'act_number', 'new_numbering',
             'folio_raw', 'folio_norm', 'verified', 'note']
CONC_COLS = [c for c in df_actes.columns if c.startswith('conc_')]
df_actes_out = df_actes[BASE_COLS + CONC_COLS].copy()

df_actes_out.to_csv(os.path.join(OUT_DIR, "actes.csv"), index=False, sep=SEP)
print(f"actes.csv → {len(df_actes_out)} lignes, {len(df_actes_out.columns)} colonnes")
df_actes_out.head(8)



Actes  chargés  :  46530 lignes  (1 ligne(s) ignorée(s))
Actes après filtre JJ096-JJ099 : 2527 lignes
volume
JJ096    386
JJ097    665
JJ098    771
JJ099    705
Name: count, dtype: int64
⚠️  4 acte(s) sans folio_sort_key :
acte_id volume folio_norm
   1387  JJ098       None
   1388  JJ098       None
   1389  JJ098       None
   1390  JJ098       None
actes.csv → 2527 lignes, 35 colonnes


,acte_id,volume,act_number,new_numbering,folio_raw,folio_norm,verified,note,conc_Longnon_act,conc_Longnon_head,conc_Longnon_locus,conc_Viard_act,conc_Viard_head,conc_Viard_locus,conc_Gascogne_act,conc_Gascogne_head,conc_Gascogne_locus,conc_Languedoc_act,conc_Languedoc_head,conc_Languedoc_locus,conc_Loire_act,conc_Loire_head,conc_Loire_locus,conc_Rouergue_act,conc_Rouergue_head,conc_Rouergue_locus,conc_IR421-JJA-JJ79A_act,conc_IR421-JJA-JJ79A_head,conc_IR421-JJA-JJ79A_locus,conc_IR422-JJ80-155_act,conc_IR422-JJ80-155_head,conc_IR422-JJ80-155_locus,conc_IR423-JJ156-211_act,conc_IR423-JJ156-211_head,conc_IR423-JJ156-211_locus
0,0,JJ096,1,NaN,6,6r,1,NaN,,,,,,,,,,,,,,,,,,,,,,1,1,,,,
1,1,JJ096,2,NaN,7,7r,1,NaN,,,,,,,,,,,,,,,,,,,,,,2,2,,,,
2,2,JJ096,3,NaN,10,10r,1,NaN,,,,,,,,,,,,,,,,,,,,,,3,3,,,,
3,3,JJ096,4,NaN,12v,12v,0,"Numéros 5 à 52 omis dans la numérotation, qui passe de 4 à 53 sans aucune lacune matérielle ou textuelle",,,,,,,,,,,,,,,,,,,,,,4,4,,,,
4,52,JJ096,53,NaN,16,16r,1,NaN,,,,,,,,,,,,,,,,,,,,,,53,53,,,,
5,53,JJ096,54,NaN,16,16r,1,NaN,,,,,,,,,,,,,,,,,,,,,,54,54,,,,
6,54,JJ096,55,NaN,16,16r,1,NaN,,,,,,,,,,,,,,,,,,,,,,55,55,,,,
7,55,JJ096,56,NaN,17v,17v,1,NaN,,,,,,,,,,,,,,,,,,,,,,56,56,,,,


## Table `zones.csv`

In [66]:
skipped = {}

with warnings.catch_warnings(record=True) as w_zones:
    warnings.simplefilter("always")
    df_zones_raw = pd.read_csv(INPUT_ZONES, sep=',', dtype=str,
                               low_memory=False, on_bad_lines='warn')
    skipped['zones'] = len(w_zones)



# Normalisation des noms de colonnes → le reste du code utilise toujours
# 'volume', 'folio_sort_key', 'register', peu importe le corpus
df_zones_raw = df_zones_raw.rename(columns={
    # Corpus JJ100-JJ139 (noms d'origine)
    'registre': 'volume',
    'ordre':    'folio_sort_key',
    # Corpus autres : 'register' et 'volume' sont déjà au bon nom,
    # mais on normalise 'register' → 'register' (no-op, au cas où)
})


def parse_labelstudio_rects(label_str):
    if pd.isna(label_str):
        return []

    if not str(label_str).strip():
        return []

    try:
        data = json.loads(label_str)
    except Exception:
        try:
            data = ast.literal_eval(label_str)
        except Exception:
            return []

    if not isinstance(data, list):
        return []

    rows = []

    for r in data:
        if 'rectanglelabels' not in r:
            continue

        rows.append({
            'class_name': r['rectanglelabels'][0],
            'x_pct': r['x'],
            'y_pct': r['y'],
            'w_pct': r['width'],
            'h_pct': r['height'],
            'image_width_px': r['original_width'],
            'image_height_px': r['original_height'],
        })

    return rows

# Supprime lignes entièrement vides ou sans annotation
df_zones_raw = df_zones_raw.dropna(how='all')
df_zones_raw = df_zones_raw.dropna(subset=['label'])
df_zones_raw = df_zones_raw[
    df_zones_raw['label'].astype(str).str.strip() != ''
]
print(f"Zones importées : {len(df_zones_raw)}")
print(f"Zones importées : {df_zones_raw.head(2)}")

df_zones_raw['url_image_full'] = df_zones_raw['image']    
# image_path contient des URLs IIIF, pas des chemins de fichiers :
# parse_image_stem ne s'applique pas ici.
# On utilise directement les colonnes 'registre' et 'ordre' déjà présentes dans df_zones_raw.







df_zones_raw['volume'] = df_zones_raw['volume'].apply(
    lambda r: re.sub(r'JJ(\d+)', lambda m: f"JJ{int(m.group(1)):03d}", str(r))
)
df_zones_raw['folio_sort_key'] = pd.to_numeric(df_zones_raw['folio_sort_key'], errors='coerce')


print(df_zones_raw)

rows = []
for _, r in df_zones_raw.iterrows():
    annotations = parse_labelstudio_rects(r['label'])
    for ann in annotations:
        rows.append({
            'url_image_full':   r['url_image_full'],
            'image_path':       r['image_path'],
            'image_filename':   r['image'],          # l'URL 1200, sert de clé de jointure
            'volume':           r['volume'],
            'folio_sort_key':   r['folio_sort_key'],
            'class_name':       ann['class_name'],
            'x_pct':            ann['x_pct'],
            'y_pct':            ann['y_pct'],
            'w_pct':            ann['w_pct'],
            'h_pct':            ann['h_pct'],
            'image_width_px':   ann['image_width_px'],
            'image_height_px':  ann['image_height_px'],
        })


df_zones = pd.DataFrame(rows)

df_zones['abs_x'] = safe_to_int(df_zones['x_pct'] / 100.0 * df_zones['image_width_px'])
df_zones['abs_y'] = safe_to_int(df_zones['y_pct'] / 100.0 * df_zones['image_height_px'])
df_zones['abs_w'] = safe_to_int(df_zones['w_pct'] / 100.0 * df_zones['image_width_px'])
df_zones['abs_h'] = safe_to_int(df_zones['h_pct'] / 100.0 * df_zones['image_height_px'])

df_zones = df_zones.sort_values(
    by=['volume', 'folio_sort_key', 'abs_y', 'abs_x'],
    ascending=[True, True, True, True]
    )

img_lookup = df_images[['image_id', 'volume', 'folio_sort_key', 
                         'folio_label', 'folio_norm']].copy()

df_zones = df_zones.merge(
    img_lookup,
    on=['volume', 'folio_sort_key'],
    how='left'
)

df_zones['zone_id'] = np.arange(1, len(df_zones) + 1)


df_zones.rename(columns={
    'registre': 'register',
    'image': 'url_image_full',
}, inplace=True)


CLASS_MAP = {

    'AC': 0,
    'AI': 1,
    'AF': 2,
    'AM': 3,
    'NIA': 4,
    'Table': 5,
}

df_zones['class_id'] = df_zones['class_name'].map(CLASS_MAP)

ZONES_COLS = [
    'zone_id', 'image_id', 'volume', 'folio_sort_key',
    'folio_label', 'folio_norm',
    'class_id', 'class_name', 
    'abs_x', 'abs_y', 'abs_w', 'abs_h',
    'url_image_full', 'image_filename',
    'image_width_px', 'image_height_px',
]
df_zones = df_zones[ZONES_COLS]

df_zones.to_csv(os.path.join(OUT_DIR, "zones.csv"), index=False, sep=SEP)

print(f"zones.csv → {len(df_zones)} lignes, {len(df_zones.columns)} colonnes")
df_zones.head(10)

# %%
# Index image_id → liste de zones triées par abs_y (ordre de lecture)
zones_by_image = defaultdict(list)

for _, row in df_zones.iterrows():
    zones_by_image[row['image_id']].append(row.to_dict())

for iid in zones_by_image:
    zones_by_image[iid].sort(key=lambda r: r['abs_y'] if pd.notna(r['abs_y']) else 0)
    
# Ensemble de tous les zone_id (pour le rapport de couverture)
all_zone_ids      = set(df_zones['zone_id'].tolist())
attributed_zone_ids = set()   # alimenté lors de la construction des liaisons

print('*' * 60)
print(f"Index zones_by_image  : {len(zones_by_image)} images ont des zones")
print(f"Total zones           : {len(all_zone_ids)}")

print('*' * 30, 'df_zones', '*' * 60)

df_zones.head(20)


Zones importées : 1549
Zones importées :   annotation_id annotator                   created_at     id  \
1         28919         1  2026-07-22T13:14:14.389845Z  49159   
2         28920         1  2026-07-22T13:14:14.389845Z  49160   

                                                                             image  \
1  https://iiif.irht.cnrs.fr/iiif/ark:/63955/v6e576jlpbid/full/1200,/0/default.jpg   
2  https://iiif.irht.cnrs.fr/iiif/ark:/63955/vxyc3rg33kh4/full/1200,/0/default.jpg   

                                                                                                     image_path  \
1  images_registres_AN_JJ035_JJ211/images/Paris_Archives_Nationales_JJ096/Paris_Archives_Nationales_JJ096_2.jpg   
2  images_registres_AN_JJ035_JJ211/images/Paris_Archives_Nationales_JJ096/Paris_Archives_Nationales_JJ096_3.jpg   

                                                                                                                     label  \
1  [{"x":0.305,"y":0.02,"width":

,zone_id,image_id,volume,folio_sort_key,folio_label,folio_norm,class_id,class_name,abs_x,abs_y,abs_w,abs_h,url_image_full,image_filename,image_width_px,image_height_px
0,1,2,JJ096,2,contre-plat sup��rieur,contre-plat sup��rieur,4,NIA,6,0,2038,2047,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/v6e576jlpbid/full/1200,/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/v6e576jlpbid/full/1200,/0/default.jpg",2048,2048
1,2,3,JJ096,3,1r,1r,5,Table,0,0,3978,6288,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vxyc3rg33kh4/full/1200,/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/vxyc3rg33kh4/full/1200,/0/default.jpg",3984,6437
2,3,4,JJ096,4,1v,1v,5,Table,0,121,3768,6103,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vbg5uy3smlz0/full/1200,/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/vbg5uy3smlz0/full/1200,/0/default.jpg",3768,6373
3,4,5,JJ096,5,2r,2r,5,Table,0,139,3928,6209,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vr66brwmaaj8/full/1200,/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/vr66brwmaaj8/full/1200,/0/default.jpg",3928,6453
4,5,6,JJ096,6,2v,2v,5,Table,150,121,3671,6177,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vnn03q4o88zo/full/1200,/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/vnn03q4o88zo/full/1200,/0/default.jpg",3824,6373
5,6,7,JJ096,7,3r,3r,5,Table,0,58,3883,6298,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/veumhm82l9ts/full/1200,/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/veumhm82l9ts/full/1200,/0/default.jpg",3888,6421
6,7,8,JJ096,8,3v,3v,5,Table,0,120,3853,6214,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vf5kcrvmb39y/full/1200,/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/vf5kcrvmb39y/full/1200,/0/default.jpg",3856,6373
7,8,9,JJ096,9,4r,4r,5,Table,0,35,3883,6214,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/v4dp81xg1muv/full/1200,/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/v4dp81xg1muv/full/1200,/0/default.jpg",3888,6325
8,9,10,JJ096,10,4v,4v,5,Table,1,23,3867,6279,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/v8bihe4q8x6d/full/1200,/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/v8bihe4q8x6d/full/1200,/0/default.jpg",3872,6397
9,10,11,JJ096,11,5r,5r,5,Table,0,32,3888,6199,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vhj4c6ggfeks/full/1200,/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/vhj4c6ggfeks/full/1200,/0/default.jpg",3888,6301


# Correction des données

## Pages non annotées

In [67]:
def find_missing_folios(df):
    missing = []

    for vol, g in df.groupby('volume'):
        folios = sorted(g['folio_sort_key'].unique())
        expected = set(range(min(folios), max(folios) + 1))
        actual = set(folios)
        gaps = sorted(expected - actual)

        if gaps:
            missing.append({
                'volume': vol,
                'missing_folios': gaps,
                'count': len(gaps)
            })

    return pd.DataFrame(missing)


df_missing = find_missing_folios(df_zones)
print(f"folios sans annotation : {len(df_missing)}")
df_missing.head(20)

folios sans annotation : 1


,volume,missing_folios,count
0,JJ096,[297],1


## Cohérence des zones

In [68]:
def y_overlap_ratio(y1, h1, y2, h2):
    """
    Renvoie le taux de recouvrement vertical entre deux zones,
    exprimé en proportion de la hauteur de la plus petite des deux.
    0.0 = pas de recouvrement, 1.0 = une zone contient totalement l'autre.
    """
    top    = max(y1, y2)
    bottom = min(y1 + h1, y2 + h2)
    overlap = max(0, bottom - top)
    if overlap == 0:
        return 0.0
    smaller_h = min(h1, h2)
    return overlap / smaller_h if smaller_h > 0 else 0.0


def find_overlapping_zones(df, threshold=0.5):
    """
    Détecte les paires de zones superposées à plus de `threshold` (défaut 50%)
    sur l'axe Y, au sein d'un même folio (volume, folio_sort_key).
    Renvoie un DataFrame avec une ligne par paire en conflit.
    """
    conflicts = []

    for (vol, folio), g in df.groupby(['volume', 'folio_sort_key']):
        g = g.sort_values('abs_y').reset_index(drop=True)
        zones = g.to_dict('records')

        for i in range(len(zones)):
            for j in range(i + 1, len(zones)):
                z1, z2 = zones[i], zones[j]
                if pd.isna(z1['abs_y']) or pd.isna(z1['abs_h']) \
                   or pd.isna(z2['abs_y']) or pd.isna(z2['abs_h']):
                    continue

                # Optimisation : si z2 commence après la fin de z1 (+marge), inutile de continuer
                if z2['abs_y'] > z1['abs_y'] + z1['abs_h']:
                    break

                ratio = y_overlap_ratio(
                    z1['abs_y'], z1['abs_h'],
                    z2['abs_y'], z2['abs_h']
                )
                if ratio > threshold:
                    conflicts.append({
                        'volume':         vol,
                        'folio_sort_key': folio,
                        'zone_id_1':      z1['zone_id'],
                        'class_name_1':   z1['class_name'],
                        'zone_id_2':      z2['zone_id'],
                        'class_name_2':   z2['class_name'],
                        'overlap_ratio':  round(ratio, 2),
                    })

    return pd.DataFrame(conflicts)


df_overlaps = find_overlapping_zones(df_zones, threshold=0.5)
print(f"Zones superposées (>50% axe Y) : {len(df_overlaps)}")
df_overlaps.head(20)

Zones superposées (>50% axe Y) : 4


,volume,folio_sort_key,zone_id_1,class_name_1,zone_id_2,class_name_2,overlap_ratio
0,JJ099,276,3376,AC,3377,AC,1.00
1,JJ099,276,3376,AC,3378,AC,0.94
2,JJ099,276,3377,AC,3378,AC,0.97
3,JJ099,276,3379,AC,3380,AC,1.00


In [69]:
def get_sequence(g):
    return g.sort_values(['abs_y', 'abs_x'])['class_name'].tolist()

def validate_sequence(seq):
    i = 0
    n = len(seq)
    if all(x == 'AC' for x in seq):
        return True
    
    if seq == ['AM']:
        return True

    if seq == ['NIA']:
        return True
    
    if seq == ['Table']:
        return True
    # --------
    # CAS 3 : (AF)? + AC* + (AI)?
    # --------
    state = "AF"

    # AF optionnel
    if i < n and seq[i] == 'AF':
        i += 1

    state = "AC"

    # AC*
    while i < n and seq[i] == 'AC':
        i += 1

    state = "AI"

    # AI optionnel
    if i < n and seq[i] == 'AI':
        i += 1

    # doit être consommé entièrement
    return i == n


def find_invalid_groups(df):
    bad = []

    for (vol, folio), g in df.groupby(['volume', 'folio_sort_key']):
        seq = get_sequence(g)

        if not validate_sequence(seq):
            bad.append({
                'volume': vol,
                'folio_sort_key': folio,
                'sequence': seq
            })

    return pd.DataFrame(bad)


df_invalid = find_invalid_groups(df_zones)

print(f"Groupes invalides : {len(df_invalid)}")
df_invalid.head(99)

Groupes invalides : 1


,volume,folio_sort_key,sequence
0,JJ097,317,"[AC, AF]"


## Cohérence transitions

In [70]:
def get_ordered_folios(df):
    """Retourne les folios ordonnés par volume et folio_sort_key."""
    return (
        df[['volume', 'folio_sort_key']]
        .drop_duplicates()
        .sort_values(['volume', 'folio_sort_key'])
        .values.tolist()
    )

def get_first_last_zones(df, vol, folio):
    """Retourne la première et dernière zone d'un folio, triées du haut vers le bas (abs_y puis abs_x)."""
    g = df[(df['volume'] == vol) & (df['folio_sort_key'] == folio)]
    seq = get_sequence(g)  # déjà trié par abs_y, abs_x dans get_sequence()
    if not seq:
        return None, None
    return seq[0], seq[-1]

# Règles : dernière zone → classes autorisées en première zone de la page suivante
TRANSITION_RULES = {
    'AC': {'AI', 'AC', 'NIA'},
    'AM': {'AM', 'AF'},
    'AI': {'AM', 'AF'},
    'AF': {'AI', 'AC', 'NIA'},
}

def find_invalid_transitions(df):
    folios = get_ordered_folios(df)
    # print(folios[1000:1010])
    bad = []

    for idx in range(len(folios) - 1):
        vol_cur,  folio_cur  = folios[idx]
        vol_next, folio_next = folios[idx + 1]

        # On ne contrôle les transitions qu'au sein d'un même volume
        if vol_cur != vol_next:
            continue

        _, last_zone  = get_first_last_zones(df, vol_cur,  folio_cur)
        first_zone, _ = get_first_last_zones(df, vol_next, folio_next)

        if last_zone is None or first_zone is None:
            continue

        allowed = TRANSITION_RULES.get(last_zone)

        # Pas de règle définie pour cette dernière zone → on ignore
        if allowed is None:
            continue

        if first_zone not in allowed:
            bad.append({
                'volume':          vol_cur,
                'folio_cur':       folio_cur,
                'folio_next':      folio_next,
                'last_zone_cur':   last_zone,
                'first_zone_next': first_zone,
                'allowed':         sorted(allowed),
            })

    return pd.DataFrame(bad)


df_invalid_transitions = find_invalid_transitions(df_zones)
print(f"Transitions invalides : {len(df_invalid_transitions)}")
df_invalid_transitions.head(99)


Transitions invalides : 1


,volume,folio_cur,folio_next,last_zone_cur,first_zone_next,allowed
0,JJ097,336,337,AI,AC,"[AF, AM]"


# Fusion des données

## Table `actes_images_zones.csv`

### Règles de liaison
- La **première zone** de chaque acte doit être **AI** ou **AC**,
  sur l'image dont le `folio_norm` correspond au folio déclaré dans le tableau des actes.
- Après une zone **AI**, on cherche séquentiellement :
  - des zones **AM** (pages entières intermédiaires, une par image)
   - puis une zone **AF** (fin de l'acte, première zone en haut d'une nouvelle page)
  - Une zone **AC** clôt immédiatement la liaison.

In [71]:

# Logs pour le rapport de qualité
log_actes_sans_folio          = []
log_actes_sans_image          = []
log_actes_sans_zone_initiale  = []
log_actes_structure_invalide  = []
log_zones_orphelines          = []

# --- Règle spécifique JJ126 : les folios image sont zéro-préfixés ("001r"),
#     contrairement aux folios acte ("1r"). On construit un index de repli
#     qui compare les folio_norm en ignorant les zéros initiaux.
def strip_leading_zeros(folio_norm):
    """'001r' -> '1r', '012bisv' -> '12bisv' (ne touche pas au suffixe r/v/bis)."""
    if not folio_norm:
        return folio_norm
    m = re.match(r'^0*(\d+)(bis)?([rv])$', folio_norm)
    if m:
        return f"{m.group(1)}{m.group(2) or ''}{m.group(3)}"
    return folio_norm

img_by_folio_norm_jj126 = defaultdict(list)
for _, row in df_images[df_images['volume'].isin(['JJ126', 'JJ181'])].iterrows():
    key = strip_leading_zeros(row['folio_norm'])
    img_by_folio_norm_jj126[key].append(row['image_id'])



    
# %%
def get_next_images(image_id, register):
    """
    Renvoie les image_id du même registre qui suivent image_id,
    dans l'ordre séquentiel du fichier images source.
    """
    reg_imgs = images_by_register.get(register, [])
    try:
        idx = reg_imgs.index(image_id)
        return reg_imgs[idx + 1:]
    except ValueError:
        return []


def make_liaison(acte_id, act_number, register, folio_norm,
                 image_id, img_order, zone, role, zone_order_global,
                 statut_zone, statut_structure=None):
    """Construit un enregistrement de liaison (dict)."""
    return {
        'acte_id':             acte_id,
        'act_number':          act_number,
        'register':            register,
        'folio_norm':          folio_norm,
        'image_id':            image_id,
        'image_order':         img_order,
        'zone_id':             zone['zone_id'] if zone else None,
        'class_id':            zone['class_id'] if zone else None,
        'class_name':          zone['class_name'] if zone else None,
        'role':                role,
        'zone_order_in_image': (int(zone['abs_y'])
                                if zone and pd.notna(zone.get('abs_y')) else None),
        'zone_order_global':   zone_order_global,
        'abs_x':               zone['abs_x'] if zone else None,
        'abs_y':               zone['abs_y'] if zone else None,
        'abs_w':               zone['abs_w'] if zone else None,
        'abs_h':               zone['abs_h'] if zone else None,
        'statut_zone':         statut_zone,
        'statut_structure':    statut_structure,
    }


def build_liaisons_for_acte(acte):
    """
    Construit la liste des enregistrements de liaison pour un acte.
    Retourne (list[dict], statut_structure).
    """
    
    liaisons   = []
    register   = acte['volume']
    folio_norm = acte['folio_norm']
    acte_id    = acte['acte_id']
    act_number = acte['act_number']

    # A — folio et registre valides
    if not folio_norm or not register or folio_norm == 'nan':
        log_actes_sans_folio.append(
            {'acte_id': acte_id, 'act_number': act_number})
        return [], 'ignoré_sans_folio'

    # B — image d'ancrage
    volume       = acte['volume']
    sort_key = int(acte['folio_sort_key']) if pd.notna(acte['folio_sort_key']) else None
    images_ancrage = img_by_folio.get((volume, sort_key), []) if sort_key is not None else []

    # Repli spécifique JJ126 et JJ181 : zéros initiaux ignorés dans la numérotation
    if not images_ancrage and volume in ('JJ126', 'JJ181'):
        key = strip_leading_zeros(folio_norm)
        images_ancrage = img_by_folio_norm_jj126.get(key, [])   

    if not images_ancrage:
        log_actes_sans_image.append({
            'acte_id': acte_id, 'act_number': act_number,
            'register': register, 'folio_norm': folio_norm})
        return [], 'avertissement_sans_image'

    roles_trouves     = []
    zone_order_global = 0

    for image_id in images_ancrage:
        zones_image = zones_by_image.get(image_id, [])

        # C — zone initiale : première AI ou AC dans l'image (ordre abs_y)
        zone_initiale = next(
            (z for z in zones_image 
             if z['class_name'] in ('AI', 'AC')
             and z['zone_id'] not in attributed_zone_ids), None)
                    
        if zone_initiale is None:
            log_actes_sans_zone_initiale.append({
                'acte_id': acte_id, 'act_number': act_number,
                'register': register, 'folio_norm': folio_norm,
                'image_id': image_id})
            liaisons.append(make_liaison(
                acte_id, act_number, register, folio_norm,
                image_id, img_order=1, zone=None, role=None,
                zone_order_global=None,
                statut_zone='ancrage_sans_zone_initiale'))
            return liaisons, 'avertissement_sans_zone_initiale'

        zone_order_global += 1
        attributed_zone_ids.add(zone_initiale['zone_id'])

        # D — cas AC : acte complet, terminé
        if zone_initiale['class_name'] == 'AC':
            roles_trouves.append('AC')
            liaisons.append(make_liaison(
                acte_id, act_number, register, folio_norm,
                image_id, img_order=1, zone=zone_initiale, role='AC',
                zone_order_global=zone_order_global,
                statut_zone='valide'))
            return liaisons, 'valide_AC'

        # D — cas AI : chercher la suite
        roles_trouves.append('AI')
        liaisons.append(make_liaison(
            acte_id, act_number, register, folio_norm,
            image_id, img_order=1, zone=zone_initiale, role='AI',
            zone_order_global=zone_order_global,
            statut_zone='valide'))

        # E — images suivantes : AM* puis AF
        img_order = 2
        for next_img_id in get_next_images(image_id, register):
            zones_suiv = zones_by_image.get(next_img_id, [])
            zones_am = [z for z in zones_suiv
                        if z['class_name'] == 'AM' and z['zone_id'] not in attributed_zone_ids]
            zones_af = [z for z in zones_suiv
                        if z['class_name'] == 'AF' and z['zone_id'] not in attributed_zone_ids]

            if zones_am and not zones_af:
                # Page entièrement couverte par l'acte (AM)
                for z_am in zones_am:   # une seule AM par image en théorie
                    zone_order_global += 1
                    attributed_zone_ids.add(z_am['zone_id'])
                    roles_trouves.append('AM')
                    liaisons.append(make_liaison(
                        acte_id, act_number, register, folio_norm,
                        next_img_id, img_order=img_order, zone=z_am, role='AM',
                        zone_order_global=zone_order_global,
                        statut_zone='valide'))
                img_order += 1

            elif zones_af:
                # Page de fin : première AF (haut de page = abs_y minimal)
                z_af = zones_af[0]
                zone_order_global += 1
                attributed_zone_ids.add(z_af['zone_id'])
                roles_trouves.append('AF')
                liaisons.append(make_liaison(
                    acte_id, act_number, register, folio_norm,
                    next_img_id, img_order=img_order, zone=z_af, role='AF',
                    zone_order_global=zone_order_global,
                    statut_zone='valide'))
                break   # AF = acte terminé

            else:
                # Ni AM ni AF : suite manquante
                liaisons.append(make_liaison(
                    acte_id, act_number, register, folio_norm,
                    next_img_id, img_order=img_order, zone=None, role=None,
                    zone_order_global=None,
                    statut_zone='zone_suite_manquante'))
                roles_trouves.append('?')
                break

    # F — valider la structure de l'acte
    if roles_trouves == ['AC']:
        statut = 'valide_AC'
    elif (roles_trouves
          and roles_trouves[0] == 'AI'
          and roles_trouves[-1] == 'AF'
          and all(r in ('AI', 'AM', 'AF') for r in roles_trouves)):
        statut = 'valide_AI_AM_AF'
    else:
        statut = 'structure_invalide'
        log_actes_structure_invalide.append({
            'acte_id':       acte_id,
            'act_number':    act_number,
            'roles_trouves': ', '.join(roles_trouves)})

    for l in liaisons:
        l['statut_structure'] = statut

    return liaisons, statut

# %%
# Détection des zones orphelines (image non trouvée lors de la jointure)
orphelines = df_zones[df_zones['image_id'].isna()]
for _, row in orphelines.iterrows():
    log_zones_orphelines.append({
        'zone_id':        row['zone_id'],
        'image_filename': row['image_filename'],
        'class_name':     row['class_name'],
    })

# %%
# Construction de toutes les liaisons
all_liaisons = []
for _, acte in df_actes.iterrows():
    liaisons, _ = build_liaisons_for_acte(acte)
    all_liaisons.extend(liaisons)

df_liaison = pd.DataFrame(all_liaisons)
if not df_liaison.empty:
    df_liaison.insert(0, 'liaison_id', range(1, len(df_liaison) + 1))

df_liaison.to_csv(os.path.join(OUT_DIR, "actes_images_zones.csv"),
                  index=False, sep=SEP)
print(f"actes_images_zones.csv → {len(df_liaison)} lignes, "
      f"{len(df_liaison.columns)} colonnes")
df_liaison.head(10)



actes_images_zones.csv → 2523 lignes, 19 colonnes


,liaison_id,acte_id,act_number,register,folio_norm,image_id,image_order,zone_id,class_id,class_name,role,zone_order_in_image,zone_order_global,abs_x,abs_y,abs_w,abs_h,statut_zone,statut_structure
0,1,0,1,JJ096,6r,13,1,12.0,1.0,AI,AI,88.0,1.0,0.0,88.0,4032.0,6294.0,valide,structure_invalide
1,2,1,2,JJ096,7r,15,1,15.0,1.0,AI,AI,1188.0,1.0,0.0,1188.0,4248.0,5196.0,valide,structure_invalide
2,3,2,3,JJ096,10r,21,1,22.0,1.0,AI,AI,5117.0,1.0,8.0,5117.0,4119.0,1240.0,valide,structure_invalide
3,4,3,4,JJ096,12v,26,1,28.0,1.0,AI,AI,5477.0,1.0,11.0,5477.0,4109.0,881.0,valide,structure_invalide
4,5,52,53,JJ096,16r,33,1,35.0,0.0,AC,AC,98.0,1.0,0.0,98.0,4150.0,3135.0,valide,None
5,6,53,54,JJ096,16r,33,1,36.0,0.0,AC,AC,3215.0,1.0,0.0,3215.0,4155.0,1841.0,valide,None
6,7,54,55,JJ096,16r,33,1,37.0,1.0,AI,AI,5050.0,1.0,0.0,5050.0,4144.0,1418.0,valide,structure_invalide
7,8,55,56,JJ096,17v,36,1,40.0,0.0,AC,AC,69.0,1.0,0.0,69.0,4024.0,6475.0,valide,None
8,9,56,57,JJ096,18r,37,1,41.0,0.0,AC,AC,88.0,1.0,0.0,88.0,4057.0,2838.0,valide,None
9,10,57,58,JJ096,18r,37,1,42.0,1.0,AI,AI,2917.0,1.0,0.0,2917.0,4057.0,3561.0,valide,structure_invalide


## Rapport de qualité

In [46]:

print("=" * 60)
print("A. CHARGEMENT")
print("=" * 60)
print(f"  Images chargées : {len(df_images_raw):>6}")
print(f"  Zones  chargées : {len(df_zones_raw):>6}")
print(f"  Actes  chargés  : {len(df_actes_raw):>6}")
print()
for fichier, n in skipped.items():
    if n:
        print(f"  ⚠ {n} ligne(s) ignorée(s) au parsing dans : {fichier}")


# %% [markdown]
# ### B — Problèmes de jointure et de localisation

# %%
print("=" * 60)
print("B. JOINTURES ET LOCALISATION")
print("=" * 60)

print(f"\n  Zones orphelines (image non trouvée) : {len(log_zones_orphelines)}")
if log_zones_orphelines:
    print(pd.DataFrame(log_zones_orphelines).to_string(index=False))

print(f"\n  Actes sans folio/registre (ignorés)  : {len(log_actes_sans_folio)}")
if log_actes_sans_folio:
    print(pd.DataFrame(log_actes_sans_folio).to_string(index=False))

print(f"\n  Actes sans image d'ancrage           : {len(log_actes_sans_image)}")
if log_actes_sans_image:
    print(pd.DataFrame(log_actes_sans_image).to_string(index=False))

print(f"\n  Actes sans zone AI/AC sur l'ancrage  : {len(log_actes_sans_zone_initiale)}")
if log_actes_sans_zone_initiale:
    print(pd.DataFrame(log_actes_sans_zone_initiale).to_string(index=False))


# %% [markdown]
# ### C — Validation des structures d'actes

# %%
print("=" * 60)
print("C. STRUCTURES D'ACTES")
print("=" * 60)

if not df_liaison.empty:
    statuts = (df_liaison.drop_duplicates('acte_id')
                         .groupby('statut_structure', dropna=False)
                         .size()
                         .reset_index(name='nb_actes'))
    print(statuts.to_string(index=False))
else:
    print("  Aucune liaison produite.")

print(f"\n  Actes à structure invalide : {len(log_actes_structure_invalide)}")
if log_actes_structure_invalide:
    print(pd.DataFrame(log_actes_structure_invalide).to_string(index=False))


# %% [markdown]
# ### D — Couverture des zones

# %%
print("=" * 60)
print("D. COUVERTURE DES ZONES")
print("=" * 60)

# Inventaire de tous les labels présents
print("\n  Labels de zones présents dans le fichier source :")
labels_counts = (df_zones.groupby('class_name', dropna=False)
                          .size()
                          .reset_index(name='nb_zones'))
labels_counts['type'] = labels_counts['class_name'].apply(
    lambda x: 'attendu' if x in LABELS_ATTENDUS else '⚠ inattendu')
print(labels_counts.to_string(index=False))

# %%
# Zones attendues (AC/AI/AM/AF) vs inattendues
LABELS_ATTENDUS  = {'AC', 'AI', 'AM', 'AF'}
LABELS_HORS_SCOPE = {'NIA', 'Table'}          # dans les paramètres en haut

df_zones_attendues   = df_zones[df_zones['class_name'].isin(LABELS_ATTENDUS)]
df_zones_hors_scope  = df_zones[df_zones['class_name'].isin(LABELS_HORS_SCOPE)]
df_zones_inattendues = df_zones[~df_zones['class_name'].isin(LABELS_ATTENDUS | LABELS_HORS_SCOPE)]



nb_att        = len(df_zones_attendues)
nb_hors_scope = len(df_zones_hors_scope)
nb_inatt      = len(df_zones_inattendues)
nb_attr       = len(df_zones_attendues[df_zones_attendues['zone_id'].isin(attributed_zone_ids)])

print(f"  Zones de label attendu   (AC/AI/AM/AF) : {nb_att}")
print(f"    → attribuées à un acte valide        : {nb_attr}")
print(f"    → NON attribuées (anomalie)           : {nb_att - nb_attr}")
print(f"\n  Zones hors scope (NIA/Table, normal)   : {nb_hors_scope}")
print(f"\n  Zones de label vraiment inattendu      : {nb_inatt}")

# %%
# Détail des zones attendues non attribuées
zones_attendues_non_attr = df_zones_attendues[
    ~df_zones_attendues['zone_id'].isin(attributed_zone_ids)
].copy()

if not zones_attendues_non_attr.empty:
    # Identifier l'acte candidat : un acte dont (register, folio_norm) matche la zone
    actes_idx = df_actes.set_index(['volume', 'folio_norm'])['acte_id'].to_dict()

    def acte_candidat(row):
        key = (row['volume'], row['folio_norm'])
        return actes_idx.get(key, 'aucun')

    zones_attendues_non_attr['acte_candidat'] = zones_attendues_non_attr.apply(
        acte_candidat, axis=1)

    # Cause probable
    def cause(row):
        if row['acte_candidat'] == 'aucun':
            return 'aucun acte ne pointe vers ce folio'
        statut_acte = None
        if not df_liaison.empty and 'acte_id' in df_liaison.columns:
            rows = df_liaison[df_liaison['acte_id'] == str(row['acte_candidat'])]
            if not rows.empty:
                statut_acte = rows.iloc[0]['statut_structure']
        if statut_acte == 'structure_invalide':
            return 'acte candidat à structure invalide'
        return 'doublon ou zone hors séquence attendue'

    zones_attendues_non_attr['cause_probable'] = zones_attendues_non_attr.apply(
        cause, axis=1)

    cols_rapport = ['zone_id', 'image_id', 'class_name', 'folio_norm',
                    'volume', 'acte_candidat', 'cause_probable']
    print("\n  ⚠ Zones attendues non attribuées (anomalies à corriger) :", len(zones_attendues_non_attr[cols_rapport]))
    print(zones_attendues_non_attr[cols_rapport][0:100].to_string(index=False))

# %%
# Détail des zones inattendues
if not df_zones_inattendues.empty:
    cols_inatt = ['zone_id', 'image_id', 'class_name', 'folio_norm',
                  'volume', 'folio_sort_key', 'url_image_full']
    print("\n  ℹ Zones de label inattendu (hors scope du modèle d'acte) :")
    print(df_zones_inattendues[cols_inatt].to_string(index=False))


# %% [markdown]
# ### E — Résumé exécutif

# %%
print("=" * 60)
print("E. RÉSUMÉ EXÉCUTIF")
print("=" * 60)

total_actes = len(df_actes)
actes_valides = 0
if not df_liaison.empty:
    actes_valides = df_liaison[
        df_liaison['statut_structure'].isin(['valide_AC', 'valide_AI_AM_AF'])
    ]['acte_id'].nunique()

taux_actes = actes_valides / total_actes * 100 if total_actes else 0
taux_zones = nb_attr / nb_att * 100 if nb_att else 0

nb_anomalies = (len(log_zones_orphelines)
                + len(log_actes_sans_image)
                + len(log_actes_sans_zone_initiale)
                + len(log_actes_structure_invalide)
                + (nb_att - nb_attr))

print(f"\n  Taux de couverture des actes  : {actes_valides}/{total_actes} "
      f"({taux_actes:.1f}%)")
print(f"  Taux de couverture des zones  : {nb_attr}/{nb_att} "
      f"({taux_zones:.1f}%) [labels AC/AI/AM/AF uniquement]")
print(f"\n  Total anomalies à traiter     : {nb_anomalies}")
print(f"    dont zones orphelines        : {len(log_zones_orphelines)}")
print(f"    dont actes sans image        : {len(log_actes_sans_image)}")
print(f"    dont actes sans zone initiale: {len(log_actes_sans_zone_initiale)}")
print(f"    dont structures invalides    : {len(log_actes_structure_invalide)}")
print(f"    dont zones attendues non attr: {nb_att - nb_attr}")
print()
print(f"  Fichiers produits dans : {os.path.abspath(OUT_DIR)}/")
print(f"    images.csv              ({len(df_images)} lignes)")
print(f"    zones.csv               ({len(df_zones)} lignes)")
print(f"    actes.csv               ({len(df_actes_out)} lignes)")
print(f"    actes_images_zones.csv  ({len(df_liaison)} lignes)")

A. CHARGEMENT
  Images chargées :   1554
  Zones  chargées :   1549
  Actes  chargés  :  46530

B. JOINTURES ET LOCALISATION

  Zones orphelines (image non trouvée) : 0

  Actes sans folio/registre (ignorés)  : 4
 acte_id  act_number
  1387.0         289
  1388.0         290
  1389.0         291
  1390.0         292

  Actes sans image d'ancrage           : 2523
 acte_id act_number register folio_norm
     0.0          1     JJ96         6r
     1.0          2     JJ96         7r
     2.0          3     JJ96        10r
     3.0          4     JJ96        12v
    52.0         53     JJ96        16r
    53.0         54     JJ96        16r
    54.0         55     JJ96        16r
    55.0         56     JJ96        17v
    56.0         57     JJ96        18r
    57.0         58     JJ96        18r
    58.0         59     JJ96        18v
    59.0         60     JJ96        18v
    60.0         61     JJ96        19r
    61.0         62     JJ96        21r
    62.0         63     JJ96       

# from csv to LabelStudio json

In [47]:
import pandas as pd
import json
import sys
from collections import defaultdict

# Lecture du CSV (séparateur tabulation)

df = df_zones_raw

df["Confidence"] = pd.to_numeric(df["Confidence"], errors="coerce")
df["Image_Width"] = pd.to_numeric(df["Image_Width"], errors="coerce")
df["Image_Height"] = pd.to_numeric(df["Image_Height"], errors="coerce")
 
# Extraction registre + numéro d'ordre depuis Image_Path
# ex: .../Paris_Archives_Nationales_JJ096_100.jpg → registre=JJ096, ordre=100
def parse_image_stem(image_path):
    stem = image_path.rsplit("/", 1)[-1].rsplit(".", 1)[0]  # retire dossier et extension
    parts = stem.split("_")
    registre = parts[-2]                  # "JJ096"
    ordre = int(parts[-1])               # 100
    return registre, ordre
 
df[["Registre", "Ordre"]] = df["Image_Path"].apply(
    lambda p: pd.Series(parse_image_stem(p))
)
 
# Tri par registre puis numéro d'ordre
df = df.sort_values(["Registre", "Ordre"]).reset_index(drop=True)
 
# Regroupement par image (en conservant l'ordre du df trié)
seen = {}
ordered_keys = []
for path in df["Image_Path"]:
    if path not in seen:
        seen[path] = True
        ordered_keys.append(path)
 
grouped = df.groupby("Image_Path", sort=False)
tasks = []
 
for image_path in ordered_keys:
    group = grouped.get_group(image_path)
    row0 = group.iloc[0]
    img_w = int(row0["Image_Width"])
    img_h = int(row0["Image_Height"])
    url_image = row0["Url_Image"].replace("/full/full/", "/full/1200,/")
 
    annotations = []
    for _, det in group.iterrows():
        # Coordonnées YOLO normalisées : cx cy w h
        parts = str(det["Detected_coordinates"]).split()
        cx, cy, bw, bh = float(parts[0]), float(parts[1]), float(parts[2]), float(parts[3])
 
        # Conversion en % (Label Studio : x,y coin supérieur gauche)
        x_pct = (cx - bw / 2) * 100
        y_pct = (cy - bh / 2) * 100
        w_pct = bw * 100
        h_pct = bh * 100
 
        annotations.append({
            "id": f"{det['Class_Id']}_{det['Class_Name']}_{round(float(det['Confidence']), 4)}",
            "type": "rectanglelabels",
            "value": {
                "x": round(x_pct, 4),
                "y": round(y_pct, 4),
                "width": round(w_pct, 4),
                "height": round(h_pct, 4),
                "rotation": 0,
                "rectanglelabels": [det["Class_Name"]]
            },
            "to_name": "image",
            "from_name": "label",
            "image_rotation": 0,
            "original_width": img_w,
            "original_height": img_h
        })
 
    task = {
        "data": {
            "image": url_image,
            "image_path": image_path,
            "registre": row0["Registre"],
            "ordre": int(row0["Ordre"]),
        },
        "annotations": [{"result": annotations}],
        "meta": {"source_file": image_path}
    }
    tasks.append(task)



output_path = "label_studio_import_JJ96-99.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(tasks, f, ensure_ascii=False, indent=2)


print(f"✓ {len(tasks)} tâche(s) générée(s) → {output_path}")

KeyError: 'Confidence'